# 04 — Product Recommendation Model

**Girdi:** `data/processed/review_text_features.parquet` & `review_concern_level.parquet`  
**Çıktı:** Eğitilmiş model + scoring tablosu + metrikler

---

### Neyi nereye kaydediyoruz?

```
CUSTOMER-RECOMMENDATION-MODEL/
│
├── data/processed/                       ← VERİ ÜRÜNLERİ (pipeline'ın bir sonraki adımının input'u)
│   ├── review_text_features.parquet        (03 notebook'tan geliyor — dokunmuyoruz)
│   ├── review_concern_level.parquet        (03 notebook'tan geliyor — dokunmuyoruz)
│   └── ml_scoring_table.parquet          ← YENİ — ürün bazlı skorlar, API'de recommend() çağrısında kullanılacak
│
├── outputs/models/                       ← EĞİTİLMİŞ MODEL DOSYALARI (inference'da yüklenir)
│   ├── final_ranker.txt (veya .json)       ranking modeli
│   ├── product_concern_embeddings.pkl      SBERT embedding'leri
│   ├── label_encoders.pkl                  categorical encoder'lar
│   ├── optuna_studies.pkl                  tuning geçmişi (opsiyonel, reproduce için)
│   └── config.json                         tüm parametreler — API bu dosyayı okuyup modeli yükler
│
├── outputs/metrics/                      ← DEĞERLENDİRME ÇIKTILARI (rapor/sunum için, production'da kullanılmaz)
│   ├── model_comparison.csv                baseline vs tuned tablo
│   ├── baseline_comparison.png             bar chart
│   ├── optuna_tuning.png                   tuning geçmişi grafiği
│   ├── baseline_vs_tuned.png               gain grafiği
│   ├── feature_importance.png              hangi feature ne kadar etkili
│   ├── shap_analysis.png                   SHAP beeswarm
│   ├── ensemble_weights.png                ağırlık pie chart
│   └── performance_heatmap.png             concern × skin_type hata analizi
```

### Pipeline Akışı

```
01_data_understanding → 02_EDA → 03_NLP_concern → [BU NOTEBOOK] → (sonraki: API & UI)
```

---
## 0. Kurulum & Import

In [ ]:
# Gerekli paketler — ilk kez çalıştırıyorsan yorum satırını kaldır
# !pip install lightgbm xgboost catboost optuna sentence-transformers scikit-learn shap

import warnings, time, json, pickle
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict

# ML
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRanker, Pool
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import ndcg_score

# Semantic
from sentence_transformers import SentenceTransformer

# Tuning
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# SHAP
import shap

SEED = 42
np.random.seed(SEED)
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']

print('✅ Import başarılı')

## 1. Dizin Yapısı

Bu notebook `notebooks/` dizininden çalışıyor → path'ler `../` ile başlıyor.

In [ ]:
# ── Path'ler (notebook notebooks/ dizininde olduğu için ../)
DATA_DIR    = Path('../data/processed')
MODELS_DIR  = Path('../outputs/models')
METRICS_DIR = Path('../outputs/metrics')

MODELS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# Kontrol — input dosyaları var mı?
for f in ['review_text_features.parquet', 'review_concern_level.parquet']:
    p = DATA_DIR / f
    status = '✅' if p.exists() else '❌ BULUNAMADI'
    print(f'  {status}  {p}')

print(f'\n  Modeller   → {MODELS_DIR.resolve()}')
print(f'  Metrikler  → {METRICS_DIR.resolve()}')

## 2. Veri Yükleme & Hızlı Bakış

In [ ]:
rtf = pd.read_parquet(DATA_DIR / 'review_text_features.parquet')
rcl = pd.read_parquet(DATA_DIR / 'review_concern_level.parquet')

print(f'review_text_features : {rtf.shape[0]:,} satır, {rtf.shape[1]} sütun')
print(f'review_concern_level : {rcl.shape[0]:,} satır, {rcl.shape[1]} sütun')
print(f'\nUnique ürün  : {rcl["product_id"].nunique():,}')
print(f'Unique concern: {rcl["concern"].nunique()}')
print(f'Unique effect : {rcl["effect_label"].nunique()}')

In [ ]:
print('── review_concern_level sütunları ──')
print(rcl.dtypes.to_string())
print()
rcl.head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1 - Concern dağılımı
c = rcl['concern'].value_counts()
axes[0].barh(c.index, c.values, color=COLORS[:len(c)])
axes[0].set_title('Concern Dağılımı', fontweight='bold')
axes[0].set_xlabel('Kayıt')

# 2 - Effect label
e = rcl['effect_label'].value_counts()
axes[1].pie(e.values, labels=e.index, autopct='%1.1f%%', colors=COLORS[:len(e)])
axes[1].set_title('Effect Label Dağılımı', fontweight='bold')

# 3 - Ürün başına review
pc = rcl.groupby('product_id').size()
axes[2].hist(pc.clip(upper=pc.quantile(0.95)), bins=40, color=COLORS[2], edgecolor='white')
axes[2].set_title('Ürün Başına Concern Kaydı', fontweight='bold')
axes[2].set_xlabel('Kayıt Sayısı')

plt.suptitle('Veri Genel Bakış', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(METRICS_DIR / 'eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Kaydedildi → outputs/metrics/eda_overview.png')

---
## 3. Feature Engineering

### Adım 3a — Aggregate Scoring

Her `(product_id, concern, skin_type)` üçlüsü için review'lardan deterministik bir skor üretiyoruz.  
Bu hem bir **baseline** hem de ensemble'ın bir katmanı.

```
effect_weight  →  helped: +1.0 | worsened: -1.5 | target_only: +0.3 | unknown: 0
weighted_score  = (rating_norm) × (effect_weight) × (concern_confidence)
aggregate_score = mean(weighted_score) + log1p(review_count) / 10
```

**Neden -1.5?** Negatif deneyim daha kesin dille yazılır, insanlar kötü deneyimi daha çok paylaşır → daha güvenilir sinyal.

In [ ]:
EFFECT_WEIGHTS = {
    'helped':      1.0,
    'worsened':   -1.5,
    'target_only': 0.3,
    'unknown':     0.0,
}


def build_aggregate_scores(rcl: pd.DataFrame, rtf: pd.DataFrame) -> pd.DataFrame:
    """(product_id, concern, skin_type) bazında aggregate skor."""
    df = rcl.copy()

    # skin_type eksikse rtf'den tamamla
    if 'skin_type' not in df.columns or df['skin_type'].isna().all():
        smap = rtf.set_index(['author_id', 'product_id'])['skin_type'].to_dict()
        df['skin_type'] = df.apply(
            lambda r: smap.get((r.get('author_id'), r['product_id']), 'Unknown'), axis=1)
    df['skin_type'] = df['skin_type'].fillna('Unknown')

    df['effect_weight']  = df['effect_label'].map(EFFECT_WEIGHTS).fillna(0.0)
    df['rating']         = pd.to_numeric(df['rating'], errors='coerce').fillna(3.0)
    df['rating_norm']    = (df['rating'] - 1) / 4
    df['weighted_score'] = df['rating_norm'] * df['effect_weight'] * df['concern_confidence']

    agg = df.groupby(['product_id', 'concern', 'skin_type']).agg(
        product_name       = ('product_name_final', 'first'),
        brand_name         = ('brand_name_final',   'first'),
        primary_category   = ('primary_category',   'first'),
        secondary_category = ('secondary_category',  'first'),
        mean_weighted_score= ('weighted_score',      'mean'),
        mean_rating        = ('rating',              'mean'),
        review_count       = ('product_id',          'count'),
        helped_count       = ('effect_label', lambda x: (x == 'helped').sum()),
        worsened_count     = ('effect_label', lambda x: (x == 'worsened').sum()),
        mean_confidence    = ('concern_confidence',   'mean'),
    ).reset_index()

    agg['review_count_bonus'] = np.log1p(agg['review_count']) / 10
    agg['helped_ratio']       = agg['helped_count']   / agg['review_count'].clip(lower=1)
    agg['worsened_ratio']     = agg['worsened_count'] / agg['review_count'].clip(lower=1)
    agg['net_effect_ratio']   = agg['helped_ratio'] - agg['worsened_ratio']
    agg['aggregate_score']    = agg['mean_weighted_score'] + agg['review_count_bonus']

    return agg


agg_scores = build_aggregate_scores(rcl, rtf)
print(f'Aggregate tablo : {agg_scores.shape[0]:,} satır')
print(f'Unique ürün     : {agg_scores["product_id"].nunique():,}')
print(f'Unique concern  : {agg_scores["concern"].nunique()}')
agg_scores.head(3)

### Adım 3b — ML Feature Matrisi

Aggregate tablodan türetilen özellikler + kategorik encode.

In [ ]:
def build_features(agg: pd.DataFrame):
    df = agg.copy()

    # ── Hedef: Relevance label (0-3, learning-to-rank) ─────────────
    conds = [
        (df['worsened_ratio'] > 0.30) | (df['helped_ratio'] < 0.10),   # 0 = kötü
        (df['helped_ratio'] >= 0.10) & (df['helped_ratio'] < 0.35),    # 1 = orta
        (df['helped_ratio'] >= 0.35) & (df['helped_ratio'] < 0.60),    # 2 = iyi
        (df['helped_ratio'] >= 0.60) & (df['mean_rating'] >= 3.5),     # 3 = çok iyi
    ]
    df['relevance_label'] = np.select(conds, [0, 1, 2, 3], default=1)

    # ── Sayısal feature'lar ────────────────────────────────────────
    df['log_review_count']    = np.log1p(df['review_count'])
    df['sqrt_review_count']   = np.sqrt(df['review_count'])
    df['rating_x_helped']     = df['mean_rating']    * df['helped_ratio']
    df['rating_x_net']        = df['mean_rating']    * df['net_effect_ratio']
    df['confidence_x_score']  = df['mean_confidence'] * df['mean_weighted_score']
    df['helped_x_confidence'] = df['helped_ratio']   * df['mean_confidence']
    df['score_x_log_reviews'] = df['mean_weighted_score'] * df['log_review_count']
    df['helped_sq']           = df['helped_ratio']  ** 2
    df['worsened_sq']         = df['worsened_ratio'] ** 2

    # ── Kategorik encode ───────────────────────────────────────────
    enc = {}
    for col in ['concern', 'skin_type', 'primary_category', 'secondary_category']:
        le = LabelEncoder()
        df[f'{col}_enc'] = le.fit_transform(df[col].fillna('Unknown').astype(str))
        enc[col] = le

    # ── Query ID — her (concern, skin_type) bir sıralama grubu ────
    df['query_id'] = df['concern_enc'].astype(str) + '_' + df['skin_type_enc'].astype(str)
    qle = LabelEncoder()
    df['query_id_enc'] = qle.fit_transform(df['query_id'])
    enc['query'] = qle

    return df, enc


ml_df, label_encoders = build_features(agg_scores)

# Feature listesi
FEATURES = [
    # Temel istatistikler
    'mean_weighted_score', 'mean_rating', 'log_review_count', 'sqrt_review_count',
    'helped_ratio', 'worsened_ratio', 'net_effect_ratio',
    'mean_confidence', 'review_count_bonus',
    # Interaction
    'rating_x_helped', 'rating_x_net', 'confidence_x_score',
    'helped_x_confidence', 'score_x_log_reviews',
    'helped_sq', 'worsened_sq',
    # Kategorik
    'concern_enc', 'skin_type_enc', 'primary_category_enc', 'secondary_category_enc',
]

CAT_IDXS = [FEATURES.index(f) for f in
            ['concern_enc', 'skin_type_enc', 'primary_category_enc', 'secondary_category_enc']]

print(f'Feature sayısı    : {len(FEATURES)}')
print(f'Categorical index : {CAT_IDXS}')
print(f'ML df shape       : {ml_df.shape}')
print(f'\nRelevance label dağılımı:')
print(ml_df['relevance_label'].value_counts().sort_index())

---
## 4. Cross-Validation Framework

**Group K-Fold**: `query_id` (concern × skin_type) grupları bölünmeden train/val'a ayrılır.  
Aynı query'nin bazı ürünleri train'de, bazıları val'da olmaz → **data leakage yok**.

In [ ]:
@dataclass
class CVResult:
    """Bir modelin CV sonuçlarını tutan küçük konteyner."""
    name: str
    ndcg5:  List[float] = field(default_factory=list)
    ndcg10: List[float] = field(default_factory=list)
    times:  List[float] = field(default_factory=list)

    @property
    def mn5(self):  return np.mean(self.ndcg5)
    @property
    def sd5(self):  return np.std(self.ndcg5)
    @property
    def mn10(self): return np.mean(self.ndcg10)
    @property
    def sd10(self): return np.std(self.ndcg10)
    @property
    def mt(self):   return np.mean(self.times)


def group_kfold(df, n=5, seed=SEED):
    """query_id_enc grubuna göre K-Fold böler."""
    uq = df['query_id_enc'].unique().copy()
    np.random.seed(seed)
    np.random.shuffle(uq)
    folds = np.array_split(uq, n)
    return [
        (df.index[df['query_id_enc'].isin(np.concatenate([folds[j] for j in range(n) if j != i]))].tolist(),
         df.index[df['query_id_enc'].isin(folds[i])].tolist())
        for i in range(n)
    ]


def calc_ndcg(df_val, preds, ks=(5, 10)):
    """Validation seti üzerinde grup bazlı NDCG hesaplar."""
    tmp = df_val.copy()
    tmp['_p'] = preds
    res = {k: [] for k in ks}
    for _, g in tmp.groupby('query_id_enc'):
        if len(g) < 2: continue
        tr = g['relevance_label'].values.reshape(1, -1)
        pr = g['_p'].values.reshape(1, -1)
        for k in ks:
            if len(g) >= k:
                try: res[k].append(ndcg_score(tr, pr, k=k))
                except: pass
    return {k: np.mean(v) if v else 0.0 for k, v in res.items()}


def gsizes(arr):
    """LightGBM/XGBoost group size listesi."""
    _, c = np.unique(arr, return_counts=True)
    return c.tolist()


N_FOLDS = 5
cv_splits = group_kfold(ml_df, n=N_FOLDS)

print(f'{N_FOLDS}-Fold Group CV hazır')
for i, (tr, va) in enumerate(cv_splits):
    print(f'  Fold {i+1}: train={len(tr):,}  val={len(va):,}')

---
## 5. Baseline Model Karşılaştırması

5 farklı modeli **default** hiperparametrelerle 5-Fold CV'den geçiriyoruz:

| Model | Tür | Neden deniyoruz? |
|-------|-----|------------------|
| **LightGBM LambdaRank** | Pairwise ranking | NDCG'yi doğrudan optimize eder, hızlı |
| **XGBoost LambdaMART** | Pairwise ranking | LightGBM alternatifi, bazen daha stabil |
| **CatBoost YetiRank** | Pairwise ranking | Categorical feature'larda güçlü |
| **Random Forest** | Pointwise regresyon | Yorumlanabilir baseline |
| **Ridge Regression** | Pointwise lineer | Hızlı, alt sınır ölçer |

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MODEL TANIMLARI
# ═══════════════════════════════════════════════════════════════════

def run_lgbm(df, splits, params=None, n_rounds=300):
    res = CVResult('LightGBM LambdaRank')
    p = {'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [5, 10],
         'learning_rate': 0.05, 'num_leaves': 31, 'min_data_in_leaf': 5,
         'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5,
         'label_gain': [0, 1, 3, 7], 'verbose': -1, 'n_jobs': -1}
    if params: p.update(params)
    for tr_i, va_i in splits:
        tr, va = df.loc[tr_i], df.loc[va_i]
        ds = lgb.Dataset(tr[FEATURES].values, label=tr['relevance_label'].values,
                         group=gsizes(tr['query_id_enc'].values))
        t0 = time.time()
        m = lgb.train(p, ds, num_boost_round=n_rounds, callbacks=[lgb.log_evaluation(9999)])
        res.times.append(time.time() - t0)
        nd = calc_ndcg(va, m.predict(va[FEATURES].values))
        res.ndcg5.append(nd[5]); res.ndcg10.append(nd[10])
    return res


def run_xgb(df, splits, params=None):
    res = CVResult('XGBoost LambdaMART')
    p = {'objective': 'rank:ndcg', 'learning_rate': 0.05, 'max_depth': 6,
         'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8,
         'n_estimators': 300, 'verbosity': 0, 'n_jobs': -1, 'random_state': SEED}
    if params: p.update(params)
    for tr_i, va_i in splits:
        tr = df.loc[tr_i].sort_values('query_id_enc')
        va = df.loc[va_i]
        m = xgb.XGBRanker(**p)
        t0 = time.time()
        m.fit(tr[FEATURES].values, tr['relevance_label'].values,
              qid=tr['query_id_enc'].values, verbose=False)
        res.times.append(time.time() - t0)
        nd = calc_ndcg(va, m.predict(va[FEATURES].values))
        res.ndcg5.append(nd[5]); res.ndcg10.append(nd[10])
    return res


def run_catboost(df, splits, params=None):
    res = CVResult('CatBoost YetiRank')
    p = {'loss_function': 'YetiRank', 'eval_metric': 'NDCG',
         'learning_rate': 0.05, 'depth': 6, 'iterations': 300,
         'random_seed': SEED, 'verbose': False}
    if params: p.update(params)
    for tr_i, va_i in splits:
        tr = df.loc[tr_i].sort_values('query_id_enc')
        va = df.loc[va_i]
        pool = Pool(tr[FEATURES].values, label=tr['relevance_label'].values,
                    group_id=tr['query_id_enc'].values, cat_features=CAT_IDXS)
        m = CatBoostRanker(**p)
        t0 = time.time(); m.fit(pool); res.times.append(time.time() - t0)
        nd = calc_ndcg(va, m.predict(va[FEATURES].values))
        res.ndcg5.append(nd[5]); res.ndcg10.append(nd[10])
    return res


def run_rf(df, splits, params=None):
    res = CVResult('Random Forest')
    p = {'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 5,
         'n_jobs': -1, 'random_state': SEED}
    if params: p.update(params)
    for tr_i, va_i in splits:
        tr, va = df.loc[tr_i], df.loc[va_i]
        m = RandomForestRegressor(**p)
        t0 = time.time(); m.fit(tr[FEATURES].values, tr['relevance_label'].values)
        res.times.append(time.time() - t0)
        nd = calc_ndcg(va, m.predict(va[FEATURES].values))
        res.ndcg5.append(nd[5]); res.ndcg10.append(nd[10])
    return res


def run_ridge(df, splits, params=None):
    res = CVResult('Ridge Regression')
    p = {'alpha': 1.0}
    if params: p.update(params)
    sc = StandardScaler()
    for tr_i, va_i in splits:
        tr, va = df.loc[tr_i], df.loc[va_i]
        Xtr = sc.fit_transform(tr[FEATURES].values)
        Xva = sc.transform(va[FEATURES].values)
        m = Ridge(**p)
        t0 = time.time(); m.fit(Xtr, tr['relevance_label'].values)
        res.times.append(time.time() - t0)
        nd = calc_ndcg(va, m.predict(Xva))
        res.ndcg5.append(nd[5]); res.ndcg10.append(nd[10])
    return res


print('Model fonksiyonları tanımlandı.')

In [ ]:
print('🏁 Baseline karşılaştırması başlıyor...\n')
baseline: Dict[str, CVResult] = {}

for tag, fn in [
    ('lgbm',     lambda: run_lgbm(ml_df, cv_splits)),
    ('xgb',      lambda: run_xgb(ml_df, cv_splits)),
    ('catboost', lambda: run_catboost(ml_df, cv_splits)),
    ('rf',       lambda: run_rf(ml_df, cv_splits)),
    ('ridge',    lambda: run_ridge(ml_df, cv_splits)),
]:
    print(f'  [{tag:<8}] ', end='', flush=True)
    baseline[tag] = fn()
    r = baseline[tag]
    print(f'NDCG@5={r.mn5:.4f}  NDCG@10={r.mn10:.4f} ± {r.sd10:.4f}  ({r.mt:.1f}s/fold)')

print('\n✅ Tamamlandı.')

In [ ]:
# Özet tablo
summary = pd.DataFrame([{
    'Model': r.name, 'NDCG@5': f'{r.mn5:.4f} ± {r.sd5:.4f}',
    'NDCG@10': f'{r.mn10:.4f} ± {r.sd10:.4f}', 'Time (s/fold)': f'{r.mt:.1f}',
} for r in baseline.values()])

print('='*72)
print('  BASELINE MODEL KARŞILAŞTIRMASI')
print('='*72)
print(summary.to_string(index=False))
print('='*72)

In [ ]:
# Grafik — bar chart + box plot
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
names = [r.name for r in baseline.values()]
order = np.argsort([r.mn10 for r in baseline.values()])[::-1]

# NDCG@5 bar
axes[0,0].barh([names[i] for i in order], [list(baseline.values())[i].mn5 for i in order],
               xerr=[list(baseline.values())[i].sd5 for i in order],
               capsize=4, color=COLORS[:5])
axes[0,0].set_title('NDCG@5 (mean ± std)', fontweight='bold')
axes[0,0].set_xlim(0, 1); axes[0,0].axvline(0.5, color='red', ls='--', alpha=0.3)

# NDCG@10 bar
axes[0,1].barh([names[i] for i in order], [list(baseline.values())[i].mn10 for i in order],
               xerr=[list(baseline.values())[i].sd10 for i in order],
               capsize=4, color=COLORS[:5])
axes[0,1].set_title('NDCG@10 (mean ± std)', fontweight='bold')
axes[0,1].set_xlim(0, 1); axes[0,1].axvline(0.5, color='red', ls='--', alpha=0.3)

# Box plots
for ax, attr, title in [(axes[1,0], 'ndcg5', 'NDCG@5 per Fold'), (axes[1,1], 'ndcg10', 'NDCG@10 per Fold')]:
    data = [getattr(r, attr) for r in baseline.values()]
    bp = ax.boxplot(data, labels=[n.split()[0] for n in names], patch_artist=True, vert=True)
    for patch, c in zip(bp['boxes'], COLORS): patch.set_facecolor(c); patch.set_alpha(0.7)
    ax.set_title(title, fontweight='bold'); ax.set_ylim(0, 1)
    ax.axhline(0.5, color='red', ls='--', alpha=0.3)

plt.suptitle('Baseline Model Karşılaştırması — 5-Fold Group CV', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(METRICS_DIR / 'baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Kaydedildi → outputs/metrics/baseline_comparison.png')

---
## 6. Optuna ile Hyperparameter Tuning

Top 3 modeli (LightGBM, XGBoost, CatBoost) Optuna TPE + MedianPruner ile optimize ediyoruz.

- **TPE Sampler:** Grid search'e göre ~10x daha verimli
- **MedianPruner:** Kötü trial'ları 1-2 fold sonra kesiyor
- **3-Fold:** Tuning sırasında hız için 3-fold kullanılır, final değerlendirme 5-fold'da yapılır

> `N_TRIALS = 50` → Production kalitesi için 150-200'e çek.

In [ ]:
N_TRIALS   = 50   # ← production için 150-200 yap
TUNE_FOLDS = 3

fast_cv = group_kfold(ml_df, n=TUNE_FOLDS)

def make_study():
    return optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1))

print(f'Tuning: {N_TRIALS} trial × 3 model × {TUNE_FOLDS}-fold CV')

In [ ]:
# ── 6a. LightGBM Tuning ───────────────────────────────────────────
def lgbm_obj(trial):
    p = {
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 15, 127),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'min_data_in_leaf':  trial.suggest_int('min_data_in_leaf', 3, 30),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq':      trial.suggest_int('bagging_freq', 1, 10),
        'lambda_l1':         trial.suggest_float('lambda_l1', 1e-4, 10.0, log=True),
        'lambda_l2':         trial.suggest_float('lambda_l2', 1e-4, 10.0, log=True),
        'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 0.5),
    }
    n_rnd = trial.suggest_int('n_rounds', 100, 600)
    scores = []
    for fi, (tr_i, va_i) in enumerate(fast_cv):
        tr, va = ml_df.loc[tr_i], ml_df.loc[va_i]
        bp = {'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [10],
              'label_gain': [0,1,3,7], 'verbose': -1, 'n_jobs': -1}
        bp.update(p)
        ds = lgb.Dataset(tr[FEATURES].values, label=tr['relevance_label'].values,
                         group=gsizes(tr['query_id_enc'].values))
        m = lgb.train(bp, ds, num_boost_round=n_rnd, callbacks=[lgb.log_evaluation(9999)])
        nd = calc_ndcg(va, m.predict(va[FEATURES].values))
        scores.append(nd[10])
        trial.report(np.mean(scores), fi)
        if trial.should_prune(): raise optuna.exceptions.TrialPruned()
    return np.mean(scores)

print(f'LightGBM tuning ({N_TRIALS} trial)...')
lgbm_study = make_study()
lgbm_study.optimize(lgbm_obj, n_trials=N_TRIALS, show_progress_bar=True)
print(f'  ✅ Best NDCG@10 = {lgbm_study.best_value:.4f}')

In [ ]:
# ── 6b. XGBoost Tuning ────────────────────────────────────────────
def xgb_obj(trial):
    p = {
        'objective': 'rank:ndcg', 'verbosity': 0, 'n_jobs': -1, 'random_state': SEED,
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
        'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
    }
    scores = []
    for fi, (tr_i, va_i) in enumerate(fast_cv):
        tr = ml_df.loc[tr_i].sort_values('query_id_enc')
        va = ml_df.loc[va_i]
        m = xgb.XGBRanker(**p)
        m.fit(tr[FEATURES].values, tr['relevance_label'].values,
              qid=tr['query_id_enc'].values, verbose=False)
        nd = calc_ndcg(va, m.predict(va[FEATURES].values))
        scores.append(nd[10])
        trial.report(np.mean(scores), fi)
        if trial.should_prune(): raise optuna.exceptions.TrialPruned()
    return np.mean(scores)

print(f'XGBoost tuning ({N_TRIALS} trial)...')
xgb_study = make_study()
xgb_study.optimize(xgb_obj, n_trials=N_TRIALS, show_progress_bar=True)
print(f'  ✅ Best NDCG@10 = {xgb_study.best_value:.4f}')

In [ ]:
# ── 6c. CatBoost Tuning ───────────────────────────────────────────
def cb_obj(trial):
    p = {
        'loss_function': 'YetiRank', 'eval_metric': 'NDCG',
        'random_seed': SEED, 'verbose': False,
        'learning_rate':       trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth':               trial.suggest_int('depth', 3, 10),
        'iterations':          trial.suggest_int('iterations', 100, 500),
        'l2_leaf_reg':         trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'border_count':        trial.suggest_int('border_count', 32, 255),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
    }
    scores = []
    for fi, (tr_i, va_i) in enumerate(fast_cv):
        tr = ml_df.loc[tr_i].sort_values('query_id_enc')
        va = ml_df.loc[va_i]
        pool = Pool(tr[FEATURES].values, label=tr['relevance_label'].values,
                    group_id=tr['query_id_enc'].values, cat_features=CAT_IDXS)
        m = CatBoostRanker(**p); m.fit(pool)
        nd = calc_ndcg(va, m.predict(va[FEATURES].values))
        scores.append(nd[10])
        trial.report(np.mean(scores), fi)
        if trial.should_prune(): raise optuna.exceptions.TrialPruned()
    return np.mean(scores)

print(f'CatBoost tuning ({N_TRIALS} trial)...')
cb_study = make_study()
cb_study.optimize(cb_obj, n_trials=N_TRIALS, show_progress_bar=True)
print(f'  ✅ Best NDCG@10 = {cb_study.best_value:.4f}')

In [ ]:
# Optuna grafikler
studies = {'LightGBM': lgbm_study, 'XGBoost': xgb_study, 'CatBoost': cb_study}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for col, (name, study) in enumerate(studies.items()):
    tdf = study.trials_dataframe()
    tdf = tdf[tdf['state'] == 'COMPLETE']

    # Row 0: trial history
    axes[0, col].plot(tdf['number'], tdf['value'], alpha=0.3, color='steelblue', lw=0.8)
    axes[0, col].plot(tdf['number'], tdf['value'].cummax(), color='crimson', lw=2, label='Running Best')
    axes[0, col].set_title(f'{name} — Trial History', fontweight='bold')
    axes[0, col].set_xlabel('Trial'); axes[0, col].set_ylabel('NDCG@10')
    axes[0, col].legend(fontsize=9)
    axes[0, col].annotate(f'Best: {study.best_value:.4f}', xy=(0.03, 0.90),
                          xycoords='axes fraction', fontsize=9,
                          bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

    # Row 1: param importance
    try:
        imp = optuna.importance.get_param_importances(study)
        idf = pd.DataFrame(list(imp.items()), columns=['p', 'v']).sort_values('v').tail(8)
        axes[1, col].barh(idf['p'], idf['v'], color='steelblue')
        axes[1, col].set_title(f'{name} — Param Importance', fontweight='bold')
    except Exception as e:
        axes[1, col].text(0.5, 0.5, str(e), ha='center', va='center', transform=axes[1,col].transAxes)

plt.suptitle('Optuna Hyperparameter Search', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(METRICS_DIR / 'optuna_tuning.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Kaydedildi → outputs/metrics/optuna_tuning.png')

---
## 7. Tuned Modeller — 5-Fold Final Değerlendirme

In [ ]:
print('Tuned modeller 5-Fold CV ile değerlendiriliyor...\n')
tuned: Dict[str, CVResult] = {}

# LightGBM tuned
lgbm_p = {k: v for k, v in lgbm_study.best_params.items() if k != 'n_rounds'}
r = run_lgbm(ml_df, cv_splits, params=lgbm_p,
             n_rounds=lgbm_study.best_params.get('n_rounds', 300))
r.name = 'LightGBM (tuned)'; tuned['lgbm'] = r
print(f'  {r.name:<26} NDCG@10 = {r.mn10:.4f} ± {r.sd10:.4f}')

# XGBoost tuned
r = run_xgb(ml_df, cv_splits, params=xgb_study.best_params)
r.name = 'XGBoost (tuned)'; tuned['xgb'] = r
print(f'  {r.name:<26} NDCG@10 = {r.mn10:.4f} ± {r.sd10:.4f}')

# CatBoost tuned
r = run_catboost(ml_df, cv_splits, params=cb_study.best_params)
r.name = 'CatBoost (tuned)'; tuned['catboost'] = r
print(f'  {r.name:<26} NDCG@10 = {r.mn10:.4f} ± {r.sd10:.4f}')

In [ ]:
# Tam karşılaştırma tablosu
rows = []
for k, r in baseline.items():
    rows.append({'Model': r.name, 'NDCG@5': r.mn5, 'NDCG@10': r.mn10,
                 'std': r.sd10, 'Type': 'Baseline', 'Gain': None})
for k, r in tuned.items():
    gain = r.mn10 - baseline[k].mn10
    rows.append({'Model': r.name, 'NDCG@5': r.mn5, 'NDCG@10': r.mn10,
                 'std': r.sd10, 'Type': 'Tuned', 'Gain': gain})

comp_df = pd.DataFrame(rows).sort_values('NDCG@10', ascending=False)
comp_df['NDCG@5']  = comp_df['NDCG@5'].map('{:.4f}'.format)
comp_df['NDCG@10_display'] = comp_df.apply(lambda r: f"{r['NDCG@10']:.4f} ± {r['std']:.4f}", axis=1)
comp_df['Gain_display'] = comp_df['Gain'].apply(lambda x: f'{x:+.4f}' if pd.notna(x) else '')

print('='*75)
print('  TÜM MODELLER — BASELINE vs TUNED')
print('='*75)
print(comp_df[['Model', 'NDCG@5', 'NDCG@10_display', 'Type', 'Gain_display']]
      .rename(columns={'NDCG@10_display': 'NDCG@10', 'Gain_display': 'Δ vs Baseline'})
      .to_string(index=False))
print('='*75)

comp_df[['Model', 'NDCG@5', 'NDCG@10', 'std', 'Type', 'Gain']].to_csv(
    METRICS_DIR / 'model_comparison.csv', index=False)
print(f'\n✅ Kaydedildi → outputs/metrics/model_comparison.csv')

In [ ]:
# Baseline vs Tuned gain grafiği
pairs = [('lgbm', 'LightGBM'), ('xgb', 'XGBoost'), ('catboost', 'CatBoost')]
x = np.arange(len(pairs)); w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b_s = [baseline[k].mn10 for k, _ in pairs]
t_s = [tuned[k].mn10    for k, _ in pairs]

b1 = ax.bar(x - w/2, b_s, w, label='Baseline', alpha=0.85, color='steelblue')
b2 = ax.bar(x + w/2, t_s, w, label='Tuned',    alpha=0.85, color='crimson')

for bar, s in zip(b1, b_s): ax.text(bar.get_x()+bar.get_width()/2, s+0.005, f'{s:.4f}', ha='center', fontsize=9)
for bar, s in zip(b2, t_s): ax.text(bar.get_x()+bar.get_width()/2, s+0.005, f'{s:.4f}', ha='center', fontsize=9, color='crimson')

ax.set_xticks(x); ax.set_xticklabels([n for _, n in pairs], fontsize=12)
ax.set_ylabel('NDCG@10'); ax.set_ylim(0, max(max(b_s), max(t_s))*1.15)
ax.set_title('Baseline vs Tuned — NDCG@10', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig(METRICS_DIR / 'baseline_vs_tuned.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Kaydedildi → outputs/metrics/baseline_vs_tuned.png')

---
## 8. Final Model Eğitimi (Tüm Veri)

En iyi tuned model seçilir ve **tüm veri** üzerinde eğitilir.

In [ ]:
BEST_KEY = max(tuned, key=lambda k: tuned[k].mn10)
print(f'🏆 En iyi tuned model: {tuned[BEST_KEY].name}')
print(f'   NDCG@10 = {tuned[BEST_KEY].mn10:.4f} ± {tuned[BEST_KEY].sd10:.4f}')
print(f'\nTüm veri üzerinde eğitiliyor...')

X_all = ml_df[FEATURES].values
y_all = ml_df['relevance_label'].values
g_all = ml_df['query_id_enc'].values

if BEST_KEY == 'lgbm':
    fp = {k: v for k, v in lgbm_study.best_params.items() if k != 'n_rounds'}
    fp.update({'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [5, 10],
              'label_gain': [0, 1, 3, 7], 'verbose': -1, 'n_jobs': -1})
    ds = lgb.Dataset(X_all, label=y_all, group=gsizes(g_all))
    final_model = lgb.train(fp, ds,
                            num_boost_round=lgbm_study.best_params.get('n_rounds', 300),
                            callbacks=[lgb.log_evaluation(9999)])
    MODEL_TYPE = 'lgbm'

elif BEST_KEY == 'xgb':
    fp = {**xgb_study.best_params, 'objective': 'rank:ndcg',
          'verbosity': 0, 'n_jobs': -1, 'random_state': SEED}
    sdf = ml_df.sort_values('query_id_enc')
    final_model = xgb.XGBRanker(**fp)
    final_model.fit(sdf[FEATURES].values, sdf['relevance_label'].values,
                    qid=sdf['query_id_enc'].values, verbose=False)
    MODEL_TYPE = 'xgb'

else:  # catboost
    fp = {**cb_study.best_params, 'loss_function': 'YetiRank',
          'eval_metric': 'NDCG', 'random_seed': SEED, 'verbose': False}
    sdf = ml_df.sort_values('query_id_enc')
    pool = Pool(sdf[FEATURES].values, label=sdf['relevance_label'].values,
                group_id=sdf['query_id_enc'].values, cat_features=CAT_IDXS)
    final_model = CatBoostRanker(**fp); final_model.fit(pool)
    MODEL_TYPE = 'catboost'

print(f'\n✅ Final model eğitildi: {MODEL_TYPE}')

In [ ]:
# Feature importance
if MODEL_TYPE == 'lgbm':    imp = final_model.feature_importance(importance_type='gain')
elif MODEL_TYPE == 'xgb':   imp = final_model.feature_importances_
else:                       imp = final_model.get_feature_importance()

imp_df = pd.DataFrame({'feature': FEATURES, 'importance': imp}).sort_values('importance')

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['crimson' if i >= len(imp_df)-5 else 'steelblue' for i in range(len(imp_df))]
ax.barh(imp_df['feature'], imp_df['importance'], color=colors)
ax.set_title(f'Feature Importance — {tuned[BEST_KEY].name}', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance (Gain)')
plt.tight_layout()
plt.savefig(METRICS_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Kaydedildi → outputs/metrics/feature_importance.png')

In [ ]:
# SHAP analizi
if MODEL_TYPE in ('lgbm', 'xgb'):
    print('SHAP hesaplanıyor...')
    sample = ml_df[FEATURES].sample(min(500, len(ml_df)), random_state=SEED).values
    explainer   = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(sample)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    plt.sca(axes[0])
    shap.summary_plot(shap_values, sample, feature_names=FEATURES, plot_type='bar', show=False)
    axes[0].set_title('SHAP — Mean Absolute', fontweight='bold')
    plt.sca(axes[1])
    shap.summary_plot(shap_values, sample, feature_names=FEATURES, show=False)
    axes[1].set_title('SHAP — Beeswarm', fontweight='bold')
    plt.tight_layout()
    plt.savefig(METRICS_DIR / 'shap_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Kaydedildi → outputs/metrics/shap_analysis.png')
else:
    print('CatBoost — SHAP yerine built-in feature importance kullanılıyor.')

---
## 9. Semantic Retrieval Katmanı (SBERT)

Her `(product_id, concern)` çifti için ortalama review embedding'i hesaplanır.  
Inference sırasında kullanıcının `"oily skin, acne"` girdisi doğal dil cümlesi olarak embed edilir ve cosine similarity ile en yakın ürünler getirilir.

In [ ]:
SBERT_NAME = 'all-MiniLM-L6-v2'
print(f'SBERT yükleniyor: {SBERT_NAME}')
sbert = SentenceTransformer(SBERT_NAME)
print('✅ Hazır')

In [ ]:
def build_product_embeddings(rcl, sbert, batch_size=128):
    """Her (product_id, concern) çifti → ortalama SBERT embedding."""
    texts = rcl['normalized_text'].fillna('').unique().tolist()
    print(f'  {len(texts):,} unique metin encode ediliyor...')
    embs = sbert.encode(texts, batch_size=batch_size,
                        show_progress_bar=True, normalize_embeddings=True)
    t2e = dict(zip(texts, embs))
    dim = embs.shape[1]

    result = {}
    for (pid, concern), grp in rcl.groupby(['product_id', 'concern']):
        vecs = np.array([t2e.get(t, np.zeros(dim)) for t in grp['normalized_text'].fillna('')])
        avg = vecs.mean(axis=0)
        n = np.linalg.norm(avg)
        result[(pid, concern)] = avg / n if n > 0 else avg

    print(f'  ✅ {len(result):,} (product, concern) embedding hazır.')
    return result


pc_embeddings = build_product_embeddings(rcl, sbert)

---
## 10. Ensemble Ağırlık Optimizasyonu

3 katmanın (aggregate, ranking model, semantic) ağırlıklarını Optuna ile optimize ediyoruz.

In [ ]:
# Model skorlarını ekle
ml_df['model_score'] = final_model.predict(ml_df[FEATURES].values)

# Grup içi min-max normalize
def gnorm(df, col):
    return df.groupby('query_id_enc')[col].transform(
        lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9))

ml_df['agg_norm']   = gnorm(ml_df, 'aggregate_score')
ml_df['model_norm'] = gnorm(ml_df, 'model_score')

# Semantic skorları hesapla
def get_sem(concern, skin):
    q = f"I have {skin} skin and I want a product that helps with {concern}."
    qe = sbert.encode([q], normalize_embeddings=True)[0]
    return {pid: float(np.dot(qe, emb)) for (pid, c), emb in pc_embeddings.items() if c == concern}

ml_df['sem_score'] = 0.0
for concern in ml_df['concern'].unique():
    for skin in ml_df['skin_type'].unique():
        mask = (ml_df['concern'] == concern) & (ml_df['skin_type'] == skin)
        if mask.sum() == 0: continue
        sem = get_sem(concern, skin)
        ml_df.loc[mask, 'sem_score'] = ml_df.loc[mask, 'product_id'].map(sem).fillna(0.0)

ml_df['sem_norm'] = gnorm(ml_df, 'sem_score')
print('✅ Tüm katman skorları hazır.')

In [ ]:
def weight_obj(trial):
    w1 = trial.suggest_float('w_agg',   0.0, 1.0)
    w2 = trial.suggest_float('w_model', 0.0, 1.0 - w1)
    w3 = max(0.0, 1.0 - w1 - w2)
    ens = w1 * ml_df['agg_norm'] + w2 * ml_df['model_norm'] + w3 * ml_df['sem_norm']
    tmp = ml_df.assign(ens=ens)
    scores = []
    for _, g in tmp.groupby('query_id_enc'):
        if len(g) >= 5:
            try:
                scores.append(ndcg_score(
                    g['relevance_label'].values.reshape(1,-1),
                    g['ens'].values.reshape(1,-1), k=10))
            except: pass
    return np.mean(scores) if scores else 0.0


print('Ensemble ağırlık optimizasyonu (200 trial)...')
w_study = optuna.create_study(direction='maximize',
                              sampler=optuna.samplers.TPESampler(seed=SEED))
w_study.optimize(weight_obj, n_trials=200, show_progress_bar=True)

W_AGG   = w_study.best_params['w_agg']
W_MODEL = w_study.best_params['w_model']
W_SEM   = max(0.0, 1.0 - W_AGG - W_MODEL)

print(f'\n✅ Optimal ağırlıklar:')
print(f'   w_aggregate : {W_AGG:.4f}')
print(f'   w_model     : {W_MODEL:.4f}')
print(f'   w_semantic  : {W_SEM:.4f}')
print(f'   Ensemble NDCG@10: {w_study.best_value:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Trial history
wdf = w_study.trials_dataframe()
wdf = wdf[wdf['state'] == 'COMPLETE']
axes[0].plot(wdf['number'], wdf['value'], alpha=0.3, color='steelblue')
axes[0].plot(wdf['number'], wdf['value'].cummax(), color='crimson', lw=2)
axes[0].set_title('Ensemble Weight Optimization', fontweight='bold')
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('NDCG@10')
axes[0].annotate(f'Best: {w_study.best_value:.4f}\nw_agg={W_AGG:.3f}  w_model={W_MODEL:.3f}  w_sem={W_SEM:.3f}',
                 xy=(0.03, 0.82), xycoords='axes fraction', fontsize=9,
                 bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

# Pie chart
vals = [W_AGG, W_MODEL, W_SEM]
lbls = ['Aggregate', tuned[BEST_KEY].name, 'Semantic']
clrs = ['steelblue', 'crimson', 'seagreen']
nz = [(v,l,c) for v,l,c in zip(vals,lbls,clrs) if v > 0.01]
axes[1].pie([v for v,_,_ in nz], labels=[l for _,l,_ in nz],
            colors=[c for _,_,c in nz], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Optimal Ensemble Ağırlıkları', fontweight='bold')

plt.tight_layout()
plt.savefig(METRICS_DIR / 'ensemble_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Kaydedildi → outputs/metrics/ensemble_weights.png')

---
## 11. Hata Analizi (Concern × Skin Type Heatmap)

In [ ]:
# Son fold üzerinde concern × skin_type performansı
_, val_idx = cv_splits[-1]
val = ml_df.loc[val_idx].copy()
val['pred'] = final_model.predict(val[FEATURES].values)

heat = []
for concern, gc in val.groupby('concern'):
    for skin, gs in gc.groupby('skin_type'):
        for _, gq in gs.groupby('query_id_enc'):
            if len(gq) >= 5:
                try:
                    n = ndcg_score(gq['relevance_label'].values.reshape(1,-1),
                                   gq['pred'].values.reshape(1,-1), k=10)
                    heat.append({'concern': concern, 'skin_type': skin, 'ndcg10': n})
                except: pass

heat_df = pd.DataFrame(heat)
if not heat_df.empty:
    pivot = heat_df.groupby(['concern', 'skin_type'])['ndcg10'].mean().unstack(fill_value=np.nan)

    fig, ax = plt.subplots(figsize=(12, 7))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn',
                vmin=0, vmax=1, linewidths=0.5, ax=ax)
    ax.set_title('NDCG@10 Performansı — Concern × Skin Type', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(METRICS_DIR / 'performance_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Kaydedildi → outputs/metrics/performance_heatmap.png')
else:
    print('Heatmap için yeterli veri yok.')

---
## 12. Her Şeyi Kaydet

```
data/processed/ml_scoring_table.parquet    ← API'nin okuyacağı ana tablo
outputs/models/final_ranker.*              ← eğitilmiş model
outputs/models/product_concern_embeddings  ← SBERT embedding'leri
outputs/models/label_encoders.pkl          ← categorical encoder'lar
outputs/models/optuna_studies.pkl          ← tuning geçmişi
outputs/models/config.json                 ← tüm parametreler
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 12a. Model dosyaları → outputs/models/
# ═══════════════════════════════════════════════════════════════════

if MODEL_TYPE == 'lgbm':
    final_model.save_model(str(MODELS_DIR / 'final_ranker.txt'))
elif MODEL_TYPE == 'xgb':
    final_model.save_model(str(MODELS_DIR / 'final_ranker.json'))
else:
    final_model.save_model(str(MODELS_DIR / 'final_ranker_catboost'))
print(f'✅ Model          → outputs/models/final_ranker.*')

with open(MODELS_DIR / 'product_concern_embeddings.pkl', 'wb') as f:
    pickle.dump(pc_embeddings, f)
print(f'✅ Embeddings     → outputs/models/product_concern_embeddings.pkl')

with open(MODELS_DIR / 'label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)
print(f'✅ Encoders       → outputs/models/label_encoders.pkl')

with open(MODELS_DIR / 'optuna_studies.pkl', 'wb') as f:
    pickle.dump({'lgbm': lgbm_study, 'xgb': xgb_study,
                 'catboost': cb_study, 'weights': w_study}, f)
print(f'✅ Optuna studies → outputs/models/optuna_studies.pkl')

# Config — API bu dosyayı okuyup modeli yükleyecek
config = {
    'best_model': tuned[BEST_KEY].name,
    'model_type': MODEL_TYPE,
    'features': FEATURES,
    'cat_feature_idxs': CAT_IDXS,
    'sbert_model': SBERT_NAME,
    'effect_weights': EFFECT_WEIGHTS,
    'ensemble_weights': {
        'w_aggregate': round(W_AGG, 4),
        'w_model':     round(W_MODEL, 4),
        'w_semantic':  round(W_SEM, 4),
    },
    'cv_scores': {
        **{f'{k}_baseline':  round(r.mn10, 4) for k, r in baseline.items()},
        **{f'{k}_tuned':     round(r.mn10, 4) for k, r in tuned.items()},
    },
    'final_ndcg10':   round(tuned[BEST_KEY].mn10, 4),
    'ensemble_ndcg10': round(w_study.best_value, 4),
    'optuna_trials':  N_TRIALS,
    'min_reviews':    3,
}

with open(MODELS_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print(f'✅ Config         → outputs/models/config.json')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 12b. ML scoring tablosu → data/processed/
#       Bu tablo API'de recommend() çağrılınca okunacak.
# ═══════════════════════════════════════════════════════════════════

ml_df.to_parquet(DATA_DIR / 'ml_scoring_table.parquet', index=False)
print(f'✅ Scoring table  → data/processed/ml_scoring_table.parquet')
print(f'   Shape: {ml_df.shape}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# DOSYA KONTROL
# ═══════════════════════════════════════════════════════════════════

print('\n📁 outputs/models/')
for p in sorted(MODELS_DIR.iterdir()):
    if p.name.startswith('.'): continue
    size = p.stat().st_size / 1024
    unit = 'KB' if size < 1024 else 'MB'
    val  = size if size < 1024 else size / 1024
    print(f'  {p.name:<45} {val:>7.1f} {unit}')

print('\n📁 outputs/metrics/')
for p in sorted(METRICS_DIR.iterdir()):
    if p.name.startswith('.'): continue
    size = p.stat().st_size / 1024
    print(f'  {p.name:<45} {size:>7.1f} KB')

print('\n📁 data/processed/ (yeni eklenen)')
p = DATA_DIR / 'ml_scoring_table.parquet'
if p.exists():
    size = p.stat().st_size / 1024
    print(f'  {p.name:<45} {size:>7.1f} KB')

---
## 13. Özet Rapor

In [ ]:
print()
print('╔' + '═'*63 + '╗')
print('║' + '  EXPERIMENT ÖZET RAPORU'.center(63) + '║')
print('╠' + '═'*63 + '╣')
print(f'║  Tarih            : {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"):<40} ║')
print(f'║  CV               : {N_FOLDS}-Fold Group K-Fold (concern × skin_type)      ║')
print(f'║  Feature sayısı   : {len(FEATURES):<40} ║')
print(f'║  Optuna trial     : {N_TRIALS} / model{"":>33}║')
print('╠' + '═'*63 + '╣')
print('║  BASELINE NDCG@10:' + ' '*44 + '║')
for k, r in sorted(baseline.items(), key=lambda x: -x[1].mn10):
    line = f'    {r.name:<28} {r.mn10:.4f} ± {r.sd10:.4f}'
    print(f'║  {line:<61}║')
print('╠' + '═'*63 + '╣')
print('║  TUNED NDCG@10:' + ' '*47 + '║')
for k, r in sorted(tuned.items(), key=lambda x: -x[1].mn10):
    d = r.mn10 - baseline[k].mn10
    line = f'    {r.name:<28} {r.mn10:.4f} ± {r.sd10:.4f}  (Δ{d:+.4f})'
    print(f'║  {line:<61}║')
print('╠' + '═'*63 + '╣')
print(f'║  🏆 FINAL MODEL    : {tuned[BEST_KEY].name:<40}║')
print(f'║     Model NDCG@10  : {tuned[BEST_KEY].mn10:<40.4f}║')
print(f'║     Ensemble NDCG  : {w_study.best_value:<40.4f}║')
print('╠' + '═'*63 + '╣')
print(f'║  ENSEMBLE AĞIRLIKLARI:' + ' '*40 + '║')
print(f'║    w_aggregate : {W_AGG:<44.4f}║')
print(f'║    w_model     : {W_MODEL:<44.4f}║')
print(f'║    w_semantic  : {W_SEM:<44.4f}║')
print('╚' + '═'*63 + '╝')

---
## Sonraki Adım → API & UI

Bu notebook'tan üretilen artefactlar bir sonraki aşamada şu şekilde kullanılacak:

```python
# API endpoint örneği (FastAPI)
# POST /recommend
# body: { "skin_type": "oily", "concern": "acne", "top_n": 10 }

@app.post('/recommend')
def recommend(skin_type: str, concern: str, top_n: int = 10):
    # 1. config.json oku → model type, features, weights
    # 2. ml_scoring_table.parquet filtrele (concern + skin_type)
    # 3. final_ranker ile rerank
    # 4. product_concern_embeddings ile semantic score
    # 5. ensemble_weights ile birleştir
    # 6. top_n döndür
```